# Auslan → English: Arm A from `how2sign_pose_only_slt.pth`

One variable against the finished `csl_stage1_weight` official-stage-3 run: the
initialisation. Everything else -- data, exclusions, optimiser, schedule,
effective batch, label smoothing, decoding -- is held identical, so the metric
difference is attributable to the starting checkpoint and nothing else.

**Why this checkpoint.** `csl_stage1_weight.pth` is a CSL pose-only base whose
decoder was trained to emit Chinese. `how2sign_pose_only_slt.pth` is already fine-tuned for
**English** sign-language translation on How2Sign (ASL, ~80 h, single signer, studio). The target language here is
English, so its decoder starts closer to the task. Uni-Sign reports BLEU-1 40.4 / BLEU-4 14.5 / ROUGE 34.3 on the How2Sign test set with this pose-only model.

**Recipe (Uni-Sign official Stage 3, single-A100 form).** Identical to
`colab_train_official.ipynb`:

- AdamW, peak lr `3e-4`, weight decay `1e-4`, β=`(0.9, 0.999)`, ε=`1e-9`, grad clip `1.0`
- cosine decay, no warmup, 20 epochs
- micro-batch `8` × gradient accumulation `4` = effective batch `32` (the paper's 4 GPUs × 8)
- pose-only, `max_length=256`, `label_smoothing=0.2`, plain beam search `num_beams=4`
- seed `0`
- `precision: bf16` -- CUDA autocast, parameters and AdamW state still FP32,
  validation decoding still FP32

**Two variables against the finished baseline, not one.** The completed
`csl_stage1_weight` run was trained in FP32; these runs are BF16, which is what
the official DeepSpeed recipe uses. So a difference against that baseline mixes
"different starting point" with "different precision". The two runs in these
notebooks are BF16 and seed 0 alike, so **How2Sign vs OpenASL stays a clean
one-variable comparison**. To make the comparison against CSL clean as well,
section 10 re-runs the CSL starting point under BF16; at roughly 3 h it is the
cheapest way to get a proper control, and it is worth doing before quoting any
"starting point X beats CSL by N BLEU" claim.

**One caveat to state in any write-up.** The paper applies this Stage-3 recipe
to a *pre-trained* checkpoint, whereas this run applies it to one that is
*already fine-tuned for SLT*. `3e-4` may partly overwrite what that decoder
learned. That is kept anyway, because changing both the initialisation and the
learning rate at once would leave the comparison uninterpretable. If this run
lands below the CSL baseline, the follow-up is a separate, clearly-labelled
lower-lr run (`1e-4`) from the same checkpoint -- not a silent edit to this one.

Verified locally before this notebook was written: all three checkpoints carry
the same 627 tensors with the same shapes, so this one is a drop-in and
`missing=0 unexpected=0` still holds.

Run the cells in order. After a Colab disconnect, re-run 1-5 and then the
training cell; `--resume` continues from the last checkpoint.

## 1. Runtime and dependencies

Use a GPU runtime. One A100 uses four accumulated micro-batches to match the paper's effective batch of 32 from four GPUs × batch 8.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'NO GPU')
DEVICE = 'cuda'

!pip -q install einops sacrebleu
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('CUDA is required for the real Uni-Sign run')

## 2. Mount Drive and locate the extracted poses

In [ ]:
import os, glob, json, copy, hashlib, shutil, signal, subprocess, sys, tarfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
MANIFEST = f'{WORK}/manifest.jsonl'
EXCLUDE = f'{WORK}/excluded.txt'
for path in (MANIFEST, EXCLUDE):
    if not os.path.exists(path):
        raise SystemExit(f'{path} is missing; run colab_setup.ipynb first')
print('work dir:', WORK)

## 3. Copy the verified project code

In [ ]:
CODE = '/content/unisign'

def locate(parts):
    for root in (DRIVE, '/content'):
        for prefix in ('', '*/', '*/*/'):
            hits = glob.glob(os.path.join(root, prefix, *parts))
            if hits:
                return hits[0]
    return None

src = locate(['unisign', 'spec.py'])
if src:
    src = os.path.dirname(src)
else:
    tarball = locate(['unisign_code.tar.gz'])
    if tarball is None:
        raise SystemExit('Upload unisign/ or unisign_code.tar.gz to Drive first')
    with tarfile.open(tarball) as tf:
        tf.extractall('/content/_code')
    src = '/content/_code/unisign'
if os.path.abspath(src) != CODE:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(src, CODE)
sys.path.insert(0, CODE)
import spec
print('code from', src)
print('spec fingerprint', spec.SCHEMA_FINGERPRINT)
assert spec.SCHEMA_FINGERPRINT == 'bc3bb2df0f22948d', 'spec.py does not match extracted poses'
_train_py = open(f'{CODE}/train.py').read()
for needed, why in [('gradient_accumulation_steps', 'gradient accumulation'),
                    ('--stop-after-epochs', 'the epoch-0 pause'),
                    ('autocast_context', 'precision: bf16')]:
    assert needed in _train_py, f'This train.py has no support for {why}; upload the current unisign_code.tar.gz first'
print('train.py supports gradient accumulation, --stop-after-epochs and bf16')

## 4. Download the pinned Uni-Sign model and mT5

Same repo commit, same mT5 revision and same Hugging Face revision as the CSL
run -- only `INIT_CKPT` differs. The sha256 below was computed from the local
copy in `checkpoints/how2sign_pose_only_slt.pth`, so a corrupted or silently re-uploaded file stops
the run instead of quietly changing the experiment.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

INIT_CKPT = 'how2sign_pose_only_slt.pth'
REPO_DIR = '/content/Uni-Sign'
REPO_COMMIT = 'eed438bcb49e30405cd6ccdfcccca330c134e830'
MT5_DIR = f'{REPO_DIR}/pretrained_weight/mt5-base'
MT5_REVISION = '2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f'
UNISIGN_REVISION = 'eab251b7fe7e8521afc0e67be98add670ea40a0d'
SHA256 = {
    'how2sign_pose_only_slt.pth': '1bfd5f3312f04e4736f0a52f4ef9535916e6de9676a2a0d00c708748683fb00d',
    'mt5-base/pytorch_model.bin': '180573b534144580f04af026da62bf71bc976ee1b7eb311b8945e2fefde8d614',
}

if not os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ZechengLi19/Uni-Sign.git', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_COMMIT], check=True)
snapshot_download('google/mt5-base', revision=MT5_REVISION, local_dir=MT5_DIR, allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
CKPT = hf_hub_download('ZechengLi19/Uni-Sign', INIT_CKPT, revision=UNISIGN_REVISION, local_dir='/content/checkpoints')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()

for name, path in [(INIT_CKPT, CKPT), ('mt5-base/pytorch_model.bin', f'{MT5_DIR}/pytorch_model.bin')]:
    if sha256(path) != SHA256[name]:
        raise SystemExit(f'{name}: sha256 mismatch')
    print('OK', name)
print('Uni-Sign code at', REPO_COMMIT[:7])

## 5. Unpack poses, gate the data, write the official-aligned config

Two gates, both of which would otherwise break comparability with the CSL run
silently:

- **Poses.** `manifest.jsonl` is the joint manifest and also lists MM-WLAuslan.
  Arm A trains on `auslandaily` only, so only those clips are required to have
  a pose; MM-WLAuslan gaps are irrelevant here and must not block this run.
- **Exclusions.** The CSL official run excluded the 131 Auslan-Daily
  mirrored/back-view uids (119 of them in train+val). The MM-WLAuslan pass
  writes to the same `excluded.txt` and had nothing to exclude, so if that file
  was overwritten this run would train on a *different* dataset than the
  baseline it is being compared to. This cell refuses to continue in that case;
  regenerate the list with `verify_pose.py --exclude-out` over the Auslan-Daily
  poses.

In [ ]:
import yaml
POSE_LOCAL = '/content/pose'
os.makedirs(POSE_LOCAL, exist_ok=True)
rows = [json.loads(line) for line in open(MANIFEST)]
ad_rows = [r for r in rows if r['dataset'] == 'auslandaily']
if not ad_rows:
    raise SystemExit('manifest.jsonl contains no auslandaily rows')
if len(glob.glob(f'{POSE_LOCAL}/*.npz')) < len(ad_rows):
    chunks = sorted(glob.glob(f'{WORK}/pose/chunk_*.tar'))
    print(f'unpacking {len(chunks)} chunk(s) from Drive ...')
    for chunk in chunks:
        with tarfile.open(chunk) as tf:
            tf.extractall(POSE_LOCAL)
present = {os.path.basename(path)[:-4] for path in glob.glob(f'{POSE_LOCAL}/*.npz')}
missing = [r['uid'] for r in ad_rows if r['uid'] not in present]
excluded = {line.strip() for line in open(EXCLUDE) if line.strip() and not line.startswith('#')}
ad_excluded = {u for u in excluded if u.startswith('ad-')}
print(f'{len(rows)} manifest rows | {len(ad_rows)} auslandaily | {len(present)} poses on disk | {len(missing)} missing | {len(ad_excluded)} auslandaily exclusions')
if missing:
    raise SystemExit(f'{len(missing)} Auslan-Daily clips have no pose, e.g. {missing[:3]}')

# The baseline this run is compared against used the Auslan-Daily exclusion
# list (131 uids). An empty or MM-WLAuslan-only list means a different dataset.
EXPECTED_AD_EXCLUSIONS = 131
if len(ad_excluded) < EXPECTED_AD_EXCLUSIONS:
    raise SystemExit(
        f'excluded.txt holds {len(ad_excluded)} ad- uids, expected {EXPECTED_AD_EXCLUSIONS}. '
        'It was probably overwritten by the MM-WLAuslan verify pass. Regenerate it with '
        'verify_pose.py --exclude-out over the Auslan-Daily poses before training, '
        'otherwise this run is not comparable to the csl_stage1_weight baseline.')

ARM = 'arm_a'
RUN = f'{ARM}__how2sign_pose_only_slt__official_stage3__single_a100__bf16'
BASE_CFG = yaml.safe_load(open(f'{CODE}/configs/{ARM}.yaml'))
CFG = copy.deepcopy(BASE_CFG)
CFG['seed'] = 0
CFG['num_workers'] = 8
CFG['log_every'] = 50
CFG['data'].update(manifest=MANIFEST, npz_dir=POSE_LOCAL, exclude=EXCLUDE, max_length=256)
CFG['backend'] = {'name': 'unisign', 'checkpoint': CKPT, 'repo': REPO_DIR, 'mt5_path': MT5_DIR, 'num_beams': 4, 'max_new_tokens': 100, 'label_smoothing': 0.2}
CFG['optim'].update(lr=3e-4, weight_decay=1e-4, batch_size=8, epochs=20, warmup_frac=0.0, grad_clip=1.0, trainable_groups=None, gradient_accumulation_steps=4, betas=[0.9, 0.999], eps=1e-9)
CFG['decode'] = {}
# BF16 autocast on CUDA; parameters and AdamW state stay FP32 and validation
# decoding stays FP32. This is part of the run fingerprint, so a run cannot be
# resumed under a different numerical mode.
CFG['precision'] = 'bf16'
CFG['output_dir'] = f'{WORK}/runs/{RUN}'
CFG_PATH = f'{WORK}/train_configs/{RUN}.yaml'
os.makedirs(os.path.dirname(CFG_PATH), exist_ok=True)
with open(CFG_PATH, 'w') as fh:
    yaml.safe_dump(CFG, fh, sort_keys=False)
print(yaml.safe_dump({'run': RUN, 'data': CFG['data'], 'backend': CFG['backend'], 'optim': CFG['optim']}, sort_keys=False))
print('effective batch:', CFG['optim']['batch_size'] * CFG['optim']['gradient_accumulation_steps'], '| precision:', CFG['precision'])

## 6. Train

The run is split in two on purpose: epoch 0 first, then epochs 1-19.

This checkpoint is already fine-tuned for English SLT, and `3e-4` with no warmup
is a large step for a model in that state -- it may overwrite the decoder that
makes this starting point worth trying at all. Epoch 0 answers that question
cheaply; the remaining 19 epochs are only worth paying for if the answer is
favourable. Under BF16 expect roughly 8-10 minutes for epoch 0 and 2.5-3 hours
for the rest, against 14 minutes and 4.3 hours in FP32 -- treat those as
estimates and read the actual seconds-per-step in the log.

This cell trains epoch 0 and stops there by itself. Section 6d, which spends the
remaining hours, is gated behind a flag, so "Run all" cannot walk past the
decision point.

**The pause changes nothing about the training.** `--stop-after-epochs` stops at
an epoch boundary and writes a checkpoint; the cosine schedule still spans all
20 epochs (`total_steps` comes from `optim.epochs`), and the flag is not part of
the resume fingerprint. `test_resume.py` covers this: pause-then-resume is
bit-identical to an uninterrupted run, on Arm A, Arm B stage 1 and Arm C.

Watch the first lines of the log for `precision=bf16`, `missing=0 unexpected=0`
and a trainable parameter count near 587.75M. Anything else means this
checkpoint did not load as the full pose-only model, or the run is not in the
precision you think it is, and it should be stopped rather than interpreted.

In [ ]:
def run_train(extra, log_path=None):
    cmd = [sys.executable, '-u', 'train.py', '--config', CFG_PATH, '--device', DEVICE] + extra
    log = open(log_path, 'a') if log_path else None
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            if log:
                log.write(line); log.flush()
    except KeyboardInterrupt:
        print('Stop pressed: train.py will save a checkpoint ...', flush=True)
        proc.send_signal(signal.SIGINT)
        for line in proc.stdout:
            print(line, end='', flush=True)
            if log:
                log.write(line); log.flush()
    proc.wait()
    if log:
        log.close()
    return proc.returncode

OUT = CFG['output_dir']
METRICS = f'{OUT}/metrics.json'
LOG = f'{OUT}/train_log.jsonl'
assert '--stop-after-epochs' in open(f'{CODE}/train.py').read(), 'Upload the train.py that has --stop-after-epochs'

# Part 1: epoch 0 only. Exit code 130 is the deliberate pause, not a failure.
if os.path.exists(METRICS):
    print('already finished:', OUT)
else:
    os.makedirs(OUT, exist_ok=True)
    rc = run_train(['--resume', '--stop-after-epochs', '1'], log_path=f'{OUT}/train.log')
    if rc not in (0, 130):
        raise RuntimeError(f'train.py exited with code {rc}; checkpoint is retained')
    print('epoch 0 done' if rc == 130 else 'run already complete')

## 6b. Read epoch 0 before paying for the rest

The comparison is the re-run Arm A official-stage-3 baseline, which averaged
**6.707** over its first epoch from `csl_stage1_weight.pth` -- a checkpoint
whose decoder had never been trained for English. Precision can still differ
between that baseline and this run; it usually moves a loss by well under 0.1,
which is small against the 0.3 band used below, but it is one more reason to
treat this cell as a cheap signal rather than a result.

- **Clearly below 6.707** (under about 6.41): the English SLT decoder survived
  the first epoch at `3e-4`. The starting point is doing work -- continue.
- **At or above 6.707** (over about 6.61): the decoder was flattened in the first epoch, and this
  run is starting from roughly where the CSL run started. Finishing it will most
  likely reproduce the baseline at four hours' cost. The better next step is a
  separate `lr=1e-4` run from the same checkpoint (`--set optim.lr=1e-4`, new
  `output_dir`), which is the deviation this experiment was designed to test.

One epoch is a cheap signal, not a result. A borderline value is a reason to
continue and look at the metrics, not to conclude anything here.

In [ ]:
import statistics
rows = [json.loads(l) for l in open(LOG)] if os.path.exists(LOG) else []
ep0 = [r['loss'] for r in rows if r['epoch'] == 0]
if not ep0:
    raise SystemExit('no epoch-0 entries in train_log.jsonl yet; run the previous cell')
BASELINE_EPOCH0 = 6.707   # re-run Arm A official stage 3, epoch-0 mean
print(f'epoch 0: {len(ep0)} logged points | first {ep0[0]:.2f} | last {ep0[-1]:.2f} '
      f'| mean {statistics.fmean(ep0):.2f}')
print(f'Arm A official stage-3 baseline, epoch 0 mean: {BASELINE_EPOCH0}')
delta = statistics.fmean(ep0) - BASELINE_EPOCH0
print(f'difference: {delta:+.2f}')
print('-> below the baseline: the English decoder survived, continue' if delta < -0.3 else
      '-> at or above the baseline: likely washed out; consider a separate lr=1e-4 run instead'
      if delta > -0.1 else '-> borderline: continue and judge on the metrics, not on this')

## 6c. Optional: search the learning rate and effective batch

Run this **instead of** finishing the current run, when epoch 0 says the
starting point was flattened -- or any time the question is "is `3e-4` at batch
32 right for a checkpoint that is already fine-tuned?". Skip it to keep the
official recipe exactly as the paper specifies; it is off by default.

**What it does.** Five short proxy runs, each 2 epochs on the real data with
everything else held at the current configuration, evaluated on a capped slice
of the validation set. A cross, not a full grid: three learning rates at the
official effective batch of 32, then the official learning rate at effective
batch 16 and 64. A full 3x3 grid would be nine runs and about 4 hours; the cross
is five runs and about 2-2.5 hours.

**The micro-batch stays 8 in every cell of the search.** Only
`gradient_accumulation_steps` changes, so the effective batch moves while the
BatchNorm statistics are computed over the same 8 samples everywhere. Changing
the micro-batch instead would move two things at once.

**Three things this search cannot tell you.**

- A 2-epoch proxy has its own cosine schedule that decays to zero inside those
  two epochs. It ranks configurations under a short budget; it does not prove
  the ranking survives 20 epochs. Treat the winner as a hypothesis to run
  properly, not as a measured result.
- Learning rate and batch size interact -- roughly, a larger batch tolerates a
  larger learning rate. If the winning learning rate is not `3e-4`, one extra
  cell at that learning rate with the winning batch is worth the 25 minutes.
- Choosing hyperparameters on the validation set and later reporting validation
  scores from the same set is tuning on your own metric. Whatever wins here has
  to be re-run in full under a new run name, and the final numbers belong on the
  test set with the decoding setting fixed in advance.

**And a config that wins here is no longer the official Stage-3 recipe.** Report
it as a separate tuned run, next to the official one -- not as a correction of
it.

In [ ]:
SEARCH = False        # set True to spend the ~2-2.5 h
PROXY_EPOCHS = 2      # short budget; see the caveats above
EVAL_LIMIT = 400      # validation clips scored per proxy run
SEARCH_GRID = [       # (lr, gradient_accumulation_steps); micro-batch stays 8
    (3e-4, 4),        # the official point: lr 3e-4, effective batch 32
    (1e-4, 4),
    (3e-5, 4),
    (3e-4, 2),        # effective batch 16
    (3e-4, 8),        # effective batch 64
]

def proxy_run(lr, accum):
    """One proxy run. Returns its output dir; resumable and skipped if done."""
    global CFG_PATH
    stem = INIT_CKPT.rsplit('.', 1)[0]
    name = f'search__{stem}__lr{lr:g}__eb{8 * accum}__bf16'
    out = f'{WORK}/runs/search/{name}'
    if os.path.exists(f'{out}/metrics.json'):
        print('already scored:', name)
        return out
    cfg = copy.deepcopy(CFG)
    cfg['optim'] = dict(CFG['optim'], lr=lr, gradient_accumulation_steps=accum,
                        epochs=PROXY_EPOCHS)
    cfg['output_dir'] = out
    path = f'{WORK}/train_configs/{name}.yaml'
    with open(path, 'w') as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)
    os.makedirs(out, exist_ok=True)
    saved, CFG_PATH = CFG_PATH, path          # run_train reads CFG_PATH
    try:
        rc = run_train(['--resume', '--eval-limit', str(EVAL_LIMIT)],
                       log_path=f'{out}/train.log')
    finally:
        CFG_PATH = saved
    if rc == 130:
        raise SystemExit('Stopped safely; re-run this cell to continue the search')
    if rc != 0:
        raise RuntimeError(f'proxy run {name} exited with code {rc}')
    return out

if not SEARCH:
    print('search not requested; set SEARCH = True to run it')
else:
    for lr, accum in SEARCH_GRID:
        print(f'=== lr={lr:g} effective_batch={8 * accum} ===', flush=True)
        proxy_run(lr, accum)
    print('search done')

### Search results

Last-epoch mean training loss and BLEU-4 on the capped validation slice, per
configuration. The two columns can disagree: loss is measured on the training
distribution and BLEU on held-out clips, and on this small a slice BLEU is
noisy. When they disagree, believe BLEU on Communication -- News on 400 capped
clips is too thin to rank anything.

In [ ]:
import statistics
stem = INIT_CKPT.rsplit('.', 1)[0]
rows = []
for lr, accum in SEARCH_GRID:
    name = f'search__{stem}__lr{lr:g}__eb{8 * accum}__bf16'
    out = f'{WORK}/runs/search/{name}'
    mpath, lpath = f'{out}/metrics.json', f'{out}/train_log.jsonl'
    if not os.path.exists(mpath):
        continue
    log = [json.loads(l) for l in open(lpath)]
    last = max(r['epoch'] for r in log)
    loss = statistics.fmean([r['loss'] for r in log if r['epoch'] == last])
    m = json.load(open(mpath))
    got = {k.split('/')[-1]: v for k, v in m.items()}
    rows.append((lr, 8 * accum, loss,
                 got.get('communication', {}).get('BLEU-4'),
                 got.get('news', {}).get('BLEU-4')))
if not rows:
    print('no finished proxy runs yet')
else:
    print(f"{'lr':>8}{'eff.batch':>11}{'loss(last ep)':>15}{'Comm BLEU-4':>13}{'News BLEU-4':>13}")
    for lr, eb, loss, c, n in rows:
        print(f'{lr:>8.0e}{eb:>11}{loss:>15.3f}'
              f"{(c if c is not None else float('nan')):>13}"
              f"{(n if n is not None else float('nan')):>13}")
    best = max((r for r in rows if r[3] is not None), key=lambda r: r[3], default=None)
    if best:
        print(f'\nbest Communication BLEU-4 at lr={best[0]:g}, effective batch={best[1]}')
        print('to run it properly: set optim.lr and optim.gradient_accumulation_steps in')
        print('section 5, give the run a NEW name (it is no longer the official recipe),')
        print('and run sections 6 and 6d as usual.')

## 6d. Epochs 1-19

Resumes from the epoch-0 checkpoint and runs to the end, then evaluates. About
2.5-3 hours on an A100 under BF16.

**This cell is gated.** `CONTINUE_TRAINING` is `False`, so "Run all" stops here
instead of spending three hours before you have read the epoch-0 number. Set it
to `True` inside the cell and run this one cell when you have decided to
continue.

`--resume` is also what you re-run after a Colab disconnect; it is safe to run
repeatedly and exits immediately once the run has finished.

In [ ]:
CONTINUE_TRAINING = False   # set True once epoch 0 has been read

if os.path.exists(METRICS):
    print('already finished:', OUT)
elif not CONTINUE_TRAINING:
    print('stopped after epoch 0.')
    print('set CONTINUE_TRAINING = True in this cell to run epochs 1-19,')
    print('or use 6c to search lr / effective batch instead.')
    print('checkpoint kept at:', f'{OUT}/checkpoints')
else:
    rc = run_train(['--resume'], log_path=f'{OUT}/train.log')
    if rc == 130:
        raise SystemExit('Stopped safely; reconnect and re-run this cell to continue')
    if rc != 0:
        raise RuntimeError(f'train.py exited with code {rc}; checkpoint is retained')
    print('metrics:', METRICS)

## 7. Loss curve and the official plain-decoding metrics

The plain row (`num_beams=4`, no decoding tricks) is the row that gets compared.
Communication and News are reported separately; there is no averaged figure.

In [ ]:
subprocess.run([sys.executable, 'plot_loss.py', f'{OUT}/train_log.jsonl'], cwd=CODE, check=True)
with open(METRICS) as fh:
    metrics = json.load(fh)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

## 8. Optional: the same weights decoded with `no_repeat_ngram_size=3`

A decoding ablation, not a retrain. It writes to `eval_nr3/` and leaves
`metrics.json` untouched. Useful only for the News looping question; it must not
replace the fixed plain row.

In [ ]:
NR3_DIR = f'{OUT}/eval_nr3'
if not os.path.exists(f'{NR3_DIR}/metrics.json'):
    rc = run_train(['--eval-only', '--eval-tag', 'nr3', '--set', 'decode.no_repeat_ngram_size=3'])
    if rc != 0:
        raise RuntimeError(f'nr3 evaluation failed with code {rc}')
else:
    print('already scored:', NR3_DIR)
if os.path.exists(f'{NR3_DIR}/metrics.json'):
    print(json.dumps(json.load(open(f'{NR3_DIR}/metrics.json')), indent=2, ensure_ascii=False))

## 9. Compare the runs

Runs that have not finished are simply absent from the table.

Read it in two steps. **How2sign bf16 vs openasl bf16** differ only in the
starting checkpoint -- that comparison is clean. **Either of them vs csl fp32**
also differs in precision; only once `csl bf16` (section 10) exists is the
comparison against the CSL starting point single-variable too.

The noise floor measured from the two old seeds is BLEU-4 0.63 on Communication
and 0.22 on News: a difference smaller than that is not a result.

In [ ]:
RUNS = [
    ('csl fp32', 'arm_a__csl_stage1_weight__official_stage3__single_a100'),
    ('csl bf16', 'arm_a__csl_stage1_weight__official_stage3__single_a100__bf16'),
    ('how2sign bf16', 'arm_a__how2sign_pose_only_slt__official_stage3__single_a100__bf16'),
    ('openasl bf16', 'arm_a__openasl_pose_only_slt__official_stage3__single_a100__bf16'),
]
TAGS = {'plain': '', 'nr3': 'eval_nr3'}
print(f"{'run':<16}{'decode':<8}{'subset':<16}{'n':>6}{'BLEU-1':>9}{'BLEU-4':>9}{'ROUGE-L':>9}{'loop%':>8}")
for init, run_name in RUNS:
    out = f'{WORK}/runs/{run_name}'
    for tag, sub in TAGS.items():
        path = os.path.join(out, sub, 'metrics.json') if sub else f'{out}/metrics.json'
        if not os.path.exists(path):
            continue
        for key, m in json.load(open(path)).items():
            print(f"{init:<16}{tag:<8}{key.split('/')[-1]:<16}{m.get('n',0):>6}"
                  f"{m.get('BLEU-1',0):>9}{m.get('BLEU-4',0):>9}{m.get('ROUGE-L',0):>9}"
                  f"{m.get('looping',0):>8}")

## 10. Optional but recommended: the BF16 CSL control

The finished CSL baseline is FP32. This cell re-runs that same starting point
under BF16, so the difference against it is attributable to the checkpoint
alone. Roughly 3 hours; it is the control that turns "X beats CSL" into a
statement with one variable in it.

It writes to its own run directory and touches nothing that already exists. Run
it once -- whichever of the two notebooks gets there first -- and the other
notebook's table will pick it up.

In [ ]:
CTRL_CKPT_NAME = 'csl_stage1_weight.pth'
CTRL_SHA = '3c81cf4a087e9e81581e57a2f33f8f0acf87b518a1ada540a660a76cdb144ced'
CTRL_RUN = 'arm_a__csl_stage1_weight__official_stage3__single_a100__bf16'
CTRL_OUT = f'{WORK}/runs/{CTRL_RUN}'
RUN_CONTROL = False   # set True to spend the ~3 h

if not RUN_CONTROL:
    print('control not requested; set RUN_CONTROL = True to run it')
elif os.path.exists(f'{CTRL_OUT}/metrics.json'):
    print('already finished:', CTRL_OUT)
else:
    ctrl_ckpt = hf_hub_download('ZechengLi19/Uni-Sign', CTRL_CKPT_NAME,
                                revision=UNISIGN_REVISION, local_dir='/content/checkpoints')
    if sha256(ctrl_ckpt) != CTRL_SHA:
        raise SystemExit('csl_stage1_weight.pth: sha256 mismatch')
    ctrl_cfg = copy.deepcopy(CFG)
    ctrl_cfg['backend'] = dict(CFG['backend'], checkpoint=ctrl_ckpt)
    ctrl_cfg['output_dir'] = CTRL_OUT
    ctrl_path = f'{WORK}/train_configs/{CTRL_RUN}.yaml'
    with open(ctrl_path, 'w') as fh:
        yaml.safe_dump(ctrl_cfg, fh, sort_keys=False)
    saved, CFG_PATH = CFG_PATH, ctrl_path      # run_train reads CFG_PATH
    try:
        os.makedirs(CTRL_OUT, exist_ok=True)
        rc = run_train(['--resume'], log_path=f'{CTRL_OUT}/train.log')
    finally:
        CFG_PATH = saved
    if rc == 130:
        raise SystemExit('Stopped safely; re-run this cell to continue')
    if rc != 0:
        raise RuntimeError(f'control run exited with code {rc}; checkpoint is retained')
    print('control metrics:', f'{CTRL_OUT}/metrics.json')

## How to read the outcome

- **Better than the CSL baseline on Communication by more than 0.63 BLEU-4**
  (baseline: 11.51): the English-SLT decoder transfers, and this checkpoint
  becomes the Arm A reference that Arm B/C must beat.
- **Better on News by more than 0.22 BLEU-4** (baseline: 2.17): the gain also
  covers the harder, high-OOV subset -- worth reporting separately, since News
  is the current bottleneck.
- **Worse or inside the noise floor**: the likely cause is `3e-4` overwriting
  the fine-tuned decoder. The follow-up is one lower-lr run (`1e-4`, same
  checkpoint, new run name), reported as a separate experiment.
- **Before quoting any of it against the CSL baseline**: check whether the
  comparison is against `csl fp32` (two variables) or `csl bf16` (one). Say
  which in the write-up.

Whatever the result, these are single-seed validation-set numbers under the
exclusion list, trained in BF16 with FP32 parameters, AdamW state and
validation decoding. Final test-set figures need a decoding setting fixed in
advance, and must not apply `excluded.txt`.

Source of the reference figures for this checkpoint: Uni-Sign (ICLR 2025),
pose-only rows. Uni-Sign reports BLEU-1 40.4 / BLEU-4 14.5 / ROUGE 34.3 on the How2Sign test set with this pose-only model.